# Model distillation: arms C and D
Run on a **T4 GPU** Colab runtime. Order: setup, teacher logits, arm C, arm D, compare.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: switch the runtime to T4'
print(torch.cuda.get_device_name(0))

In [ ]:
import os
REPO = 'distillation-appliance'
if not os.path.exists(REPO):
    !git clone https://github.com/Nethmini-Rathnayake/distillation-appliance.git
%cd {REPO}
!git pull
!pip install -q -e .

In [ ]:
!python -m factory.cli --config configs/arm_c.yaml

## Arm C: student, hard labels

In [ ]:
!python -m factory.train --config configs/arm_c.yaml

## Teacher logits (needed by arm D)
Fine-tunes roberta-large, gates on validation accuracy, then writes `data/teacher_logits.parquet`. The frozen eval set is never used.

In [ ]:
!python -m factory.teacher --config configs/arm_d.yaml

## Arm D: student, distilled

In [ ]:
!python -m factory.train --config configs/arm_d.yaml

## Compare

In [ ]:
import json, glob, pandas as pd
rows = []
for f in sorted(glob.glob('runs/*/metrics.json')):
    m = json.load(open(f)); rows.append({'run': f.split('/')[1], **{k: m.get(k) for k in ['accuracy','macro_f1','ece','latency_p50_ms','latency_p95_ms','adjusted_latency_p50_ms','adjusted_latency_p95_ms','input_tokens','model_size_bytes','peak_memory_bytes']}})
pd.DataFrame(rows)